# Verilog Fine-tuning Pipeline -- Llama-3.2-1B-Instruct + QLoRA

This notebook fine-tunes a 1-billion-parameter language model to generate Verilog hardware description code using **QLoRA** -- a memory-efficient variant of LoRA that quantizes the base model to 4-bit. It demonstrates the complete pipeline: dataset construction -> QLoRA fine-tuning -> baseline vs. fine-tuned evaluation.

---

## What is QLoRA?

**QLoRA** [(Dettmers et al., 2023)](https://arxiv.org/abs/2305.14314) combines two ideas:

### 1. NF4 Quantization (NormalFloat4)

Standard 4-bit quantization assigns uniform bins across the weight range. NF4 instead uses bins spaced according to a **standard normal distribution** -- which matches how neural network weights are actually distributed after training. This minimizes quantization error for the same bit-width.

Each weight block (e.g. 64 weights) gets a shared scale factor stored in float32. The weights themselves are stored as 4-bit indices into the 16-bin NF4 lookup table.

### 2. Double Quantization

The float32 scale factors are themselves quantized to 8-bit, saving an additional ~0.5 GB for a 1B-parameter model.

### How training works

Even though weights are stored in 4-bit, **all computation happens in float16/bfloat16**. Before each matrix multiply, weights are dequantized on-the-fly to fp16, the matmul runs, and results flow into the LoRA adapter update. The base weights are never updated -- only the LoRA A and B matrices (stored in fp16).

```
  NF4 weight -> dequantize to fp16 -> matmul -> output
                                                   |
                           LoRA A x B (fp16) ------+ (added during forward pass)
```

### LoRA vs. QLoRA vs. Full Fine-tuning

| Method | VRAM (1B model) | Quality | Speed |
|--------|----------------|---------|-------|
| Full fine-tune | ~24 GB | Best | Slowest |
| LoRA (fp16) | ~8-10 GB | Very good | Fast |
| **QLoRA (4-bit NF4)** | ~4-5 GB | Good | Fastest |

QLoRA's quality is usually within 1-3 percentage points of full-precision LoRA -- a good tradeoff for free-tier Colab GPUs.

---

## Before running
1. Set runtime to GPU: `Runtime -> Change runtime type -> T4 GPU`
2. Upload your repo zip (Cell 1) -- must include `generated_tests/`, `VerilogDB/`, and all `.py` scripts
3. Add your HuggingFace token to Colab Secrets: key icon on the left sidebar -> name it `HF_TOKEN`

## Pipeline overview
```
VerilogDB/            <- 38K module spec JSON files
generated_tests/      <- AI-generated testbenches (*_tb.v)
       |
  Cell 5: Build data.jsonl + train/test split
       |
  Cell 6: Fine-tune Llama-3.2-1B with QLoRA (NF4 4-bit)
       |
  Cell 7: Generate Verilog (baseline + finetuned)
       |
  Cell 9: Evaluate with iverilog -> pass@1 + failure breakdown
```

---
## Section 1 -- Environment Setup

Cells 1-4 prepare the Colab environment: upload data, install packages, authenticate, and verify GPU.

In [ ]:
# -- Cell 1: Upload and extract project zip --
from google.colab import files
import zipfile, os

print('Upload your project zip (must include generated_tests/, VerilogDB/, *.py)')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/project')

os.chdir('/content/project')
print('Working directory:', os.getcwd())
print(os.listdir('.'))

### Dependencies

- **[Unsloth](https://github.com/unslothai/unsloth)** -- patches Hugging Face `transformers` and `peft` with faster attention kernels and memory-efficient cross-entropy. It also bundles [bitsandbytes](https://github.com/TimDettmers/bitsandbytes), which provides the NF4 quantization and 8-bit optimizer kernels that QLoRA relies on.
- **iverilog** -- Icarus Verilog, an open-source Verilog simulator used to compile and simulate testbenches to check functional correctness of generated Verilog.

> **bitsandbytes vs. Unsloth:** bitsandbytes handles the actual NF4 storage and dequantization on GPU. Unsloth patches the attention layers and training loop for speed -- it does not replace bitsandbytes for the quantization itself.

In [ ]:
# -- Cell 2: Install dependencies --
# Unsloth pins compatible torch/transformers/trl/peft/bitsandbytes versions.
%%capture
!pip install unsloth
!apt-get install -y -q iverilog

In [ ]:
# -- Cell 3: HuggingFace login --
# Llama-3.2 is a gated model -- accept Meta's license on HuggingFace first.
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print('Logged in to HuggingFace')

### GPU and VRAM

QLoRA's VRAM budget for Llama-3.2-1B on a T4 (16 GB):
- Base model (NF4 4-bit): ~0.5 GB  (1B params x 0.5 bytes)
- Dequantization buffers (fp16 workspace): ~1-2 GB
- LoRA adapter weights (fp16): ~50 MB  (r=16 across 7 layers)
- Optimizer states (8-bit Adam for adapter params only): ~0.1 GB
- Activations + gradients: ~1-2 GB
- **Total: ~3-5 GB peak -- fits comfortably on T4**

Contrast with fp16 LoRA (~10 GB peak) -- QLoRA frees ~5-7 GB, leaving room for larger batch sizes or longer sequences.

> **If you hit OOM despite QLoRA:** The most likely cause is a very long training example (>2048 tokens). Reduce `MAX_SEQ_LENGTH` to 1024 or reduce `BATCH_SIZE` to 2.

In [ ]:
# -- Cell 4: Check GPU + verify data --
import torch
from pathlib import Path

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    total_vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print('VRAM:', total_vram, 'GB')
    print('QLoRA needs ~4-5 GB -- T4 (16 GB) is more than sufficient.')
else:
    print('NO GPU -- go to Runtime > Change runtime type and select T4 GPU')

tb_files = list(Path('generated_tests').glob('*_tb.v'))
print(f'Testbenches found: {len(tb_files)}')
print(f'VerilogDB JSONs found: {len(list(Path("VerilogDB").glob("*.json")))}')

---
## Section 2 -- Dataset Construction

### Why instruction fine-tuning?

We're doing **supervised fine-tuning (SFT)** in instruction-following format. Each training example is a user/assistant message pair:

```
<|user|>
Generate a Verilog module named `adder` with the following specification.

Description: 4-bit ripple-carry adder.

Ports:
  input [3:0] a
  input [3:0] b
  output [4:0] sum

Respond with only the complete Verilog module code.
<|assistant|>
module adder(input [3:0] a, input [3:0] b, output [4:0] sum);
  assign sum = a + b;
endmodule
```

The model learns to map **specification -> code**. The loss is computed only over the assistant tokens (the Verilog code), not the user prompt.

### Train/test split

We use an 80/20 random split. The test IDs are written to `data/test_ids.txt` so that baseline and fine-tuned inference run on the exact same problems -- essential for a fair comparison.

> **Why shuffle before splitting?** VerilogDB modules are organized by type. Without shuffling, the test set might be dominated by one category, making the evaluation unrepresentative.

In [ ]:
# -- Cell 5: Build dataset (data.jsonl + train/test split) --
import json
import random
from pathlib import Path

GENERATED_TESTS_DIR = Path('generated_tests')
VERILOG_DB = Path('VerilogDB')
DATA_DIR = Path('data')
DATA_JSONL = Path('data.jsonl')
SPLIT_RATIO = 0.8  # 80% train, 20% test
SEED = 42          # fixed seed for reproducibility


def build_prompt(spec):
    module_name = spec.get('module_name', 'unknown')
    description = spec.get('description', '').strip()
    ports = spec.get('ports', [])
    port_lines = []
    for p in ports:
        w = p.get('bit_width', 1)
        width_str = f'[{w-1}:0] ' if isinstance(w, int) and w > 1 else ''
        port_lines.append(f"  {p['direction']} {width_str}{p['name']}")
    port_str = '\n'.join(port_lines) if port_lines else '  (no ports listed)'
    return (
        f'Generate a Verilog module named `{module_name}` with the following specification.\n\n'
        f'Description: {description}\n\n'
        f'Ports:\n{port_str}\n\n'
        'Respond with only the complete Verilog module code. Do not include markdown fences or any explanation.'
    )


tb_files = sorted(GENERATED_TESTS_DIR.glob('*_tb.v'))
print(f'Found {len(tb_files)} testbenches. Building dataset...')

records, skipped = [], 0
for tb_path in tb_files:
    module_id = tb_path.stem.removesuffix('_tb')
    spec_path = VERILOG_DB / f'{module_id}.json'
    if not spec_path.exists():
        print(f'  SKIP {module_id}: spec not found in VerilogDB')
        skipped += 1
        continue
    spec = json.loads(spec_path.read_text(encoding='utf-8'))
    verilog_code = spec.get('verilog_code', '').strip()
    if not verilog_code:
        print(f'  SKIP {module_id}: no verilog_code in spec')
        skipped += 1
        continue
    records.append({
        'id': module_id,
        'messages': [
            {'role': 'user',      'content': build_prompt(spec)},
            {'role': 'assistant', 'content': verilog_code},
        ],
    })

print(f'Built {len(records)} records ({skipped} skipped).')

DATA_DIR.mkdir(exist_ok=True)
with DATA_JSONL.open('w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec) + '\n')

random.seed(SEED)
shuffled = records.copy()
random.shuffle(shuffled)
split_idx = int(len(shuffled) * SPLIT_RATIO)
train_records = shuffled[:split_idx]
test_records  = shuffled[split_idx:]

with (DATA_DIR / 'train.jsonl').open('w', encoding='utf-8') as f:
    for rec in train_records:
        f.write(json.dumps(rec) + '\n')
with (DATA_DIR / 'test.jsonl').open('w', encoding='utf-8') as f:
    for rec in test_records:
        f.write(json.dumps(rec) + '\n')

split_meta = {
    'total': len(records),
    'train': len(train_records),
    'test':  len(test_records),
    'split_ratio': SPLIT_RATIO,
    'seed': SEED,
    'train_ids': [r['id'] for r in train_records],
    'test_ids':  [r['id'] for r in test_records],
}
(DATA_DIR / 'split.json').write_text(json.dumps(split_meta, indent=2), encoding='utf-8')
(DATA_DIR / 'test_ids.txt').write_text('\n'.join(split_meta['test_ids']) + '\n', encoding='utf-8')

print(f"Total: {split_meta['total']}  Train: {split_meta['train']}  Test: {split_meta['test']}")

---
## Section 3 -- QLoRA Fine-tuning

### Key hyperparameters explained

| Hyperparameter | Value | Why |
|---|---|---|
| `BASE_MODEL` | `unsloth/Llama-3.2-1B-Instruct-bnb-4bit` | Pre-quantized to NF4 by Unsloth -- avoids quantizing on-the-fly at load time. |
| `load_in_4bit` | `True` | Activates NF4 quantization. Base weights stored at ~0.5 bytes/param instead of 2 bytes/param (fp16). |
| `dtype` | `None` | Unsloth auto-selects: bfloat16 on Ampere+, float16 otherwise. Adapters and activations use this dtype. |
| `LORA_R` | 16 | Rank of the adapter matrices. Same as the fp16 LoRA notebook for an apples-to-apples comparison. |
| `LORA_ALPHA` | 16 | Scaling factor. alpha/r = 1.0 keeps weight updates at unit scale. |
| `target_modules` | all 7 projections | Adapts all attention and MLP layers for maximum expressiveness. |
| `BATCH_SIZE` | 4 | Per-device. Effective batch = 4 x gradient_accumulation_steps = 16. |
| `gradient_accumulation_steps` | 4 | Accumulates gradients over 4 steps, simulating a larger batch. |
| `LR` | 2e-4 | Adapters start from random init, so a larger LR than pre-training is appropriate. |
| `lr_scheduler_type` | cosine | Smooth LR decay to near-zero. Prevents overshooting near convergence. |
| `warmup_ratio` | 0.1 | Linear LR warmup for first 10% of steps. |
| `optim` | `adamw_8bit` | bitsandbytes 8-bit Adam: momentum and variance stored in 8-bit on GPU. Only adapter params are optimized (tiny), so absolute savings are small but it avoids CPU paging overhead. |
| `fp16` | `True` | Fixed fp16 for adapter weights and activations in QLoRA. |

### adamw_8bit vs. paged_adamw_32bit

The fp16 LoRA notebook uses `paged_adamw_32bit` (32-bit optimizer states paged to CPU when idle). QLoRA uses `adamw_8bit` (8-bit quantized optimizer states, always on GPU). Since QLoRA's adapter is tiny (~50 MB for r=16), optimizer state memory is negligible either way. `adamw_8bit` is the conventional QLoRA choice and avoids CPU-GPU transfer overhead.

### Expected loss curve

The loss is cross-entropy over the assistant tokens (Verilog code) only:
- Initial loss: 2.0-3.5 (random adapter init)
- After 1 epoch: 0.8-1.5
- After 3 epochs: 0.4-0.9

QLoRA loss curves may be slightly noisier than fp16 LoRA due to quantization noise in the forward pass. This is expected and does not indicate a problem.

In [ ]:
# -- Cell 6: Fine-tune with QLoRA (4-bit NF4 + LoRA fp16) --
# OOM? Reduce BATCH_SIZE to 2.
import json
from pathlib import Path
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel

# Pre-quantized NF4 model -- avoids quantizing at load time
BASE_MODEL = 'unsloth/Llama-3.2-1B-Instruct-bnb-4bit'
FINETUNE_DATA = 'data/train.jsonl'
OUTPUT_DIR = Path('output/llama3_2_1b_qlora_finetune')
MAX_SEQ_LENGTH = 2048    # examples longer than this are truncated

# LoRA adapter config (same as fp16 LoRA notebook for fair comparison)
LORA_R = 16
LORA_ALPHA = 16          # alpha/r = 1.0 -- unit-scale updates
LORA_TARGET_MODULES = [  # all attention + MLP projections
    'q_proj', 'k_proj', 'v_proj',
    'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]

EPOCHS = 3
BATCH_SIZE = 4           # effective batch = 4 x 4 = 16
LR = 2e-4

# Load pre-quantized 4-bit model
# dtype=None -> Unsloth auto-selects bf16 on Ampere+, fp16 otherwise
print(f'Loading {BASE_MODEL} in 4-bit NF4...')
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,   # activates NF4 quantization for base weights
)

# Inject LoRA adapters in fp16 on top of the frozen 4-bit base
ft_model = FastLanguageModel.get_peft_model(
    ft_model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,                        # 0 recommended for SFT
    bias='none',
    use_gradient_checkpointing='unsloth',  # recompute activations to save VRAM
    random_state=42,
)

# Load training data
ft_records = []
with open(FINETUNE_DATA, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            ft_records.append(json.loads(line))
print(f'Loaded {len(ft_records)} training examples')
ft_dataset = Dataset.from_list(ft_records)

# Apply Llama-3 chat template: converts message dicts -> single formatted string
def ft_formatting_func(examples):
    return {'text': [
        ft_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in examples['messages']
    ]}

ft_dataset = ft_dataset.map(ft_formatting_func, batched=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

trainer = SFTTrainer(
    model=ft_model,
    processing_class=ft_tokenizer,
    train_dataset=ft_dataset,
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        output_dir=str(OUTPUT_DIR),
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=4,       # effective batch = 16
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type='cosine',          # smooth decay to ~0
        warmup_ratio=0.1,                    # linear warmup for first 10% of steps
        fp16=True,                           # adapters + activations in float16
        logging_steps=10,
        save_strategy='epoch',
        optim='adamw_8bit',                  # 8-bit quantized Adam (bitsandbytes)
        seed=42,
        report_to='none',
    ),
)

import time
print('Starting training...')
t_train_start = time.time()
trainer.train()
t_train_elapsed = time.time() - t_train_start
t_h, t_rem = divmod(int(t_train_elapsed), 3600)
t_m, t_s  = divmod(t_rem, 60)
print(f'Training finished in {t_h:02d}:{t_m:02d}:{t_s:02d} ({t_train_elapsed:.1f}s total, {t_train_elapsed/EPOCHS:.1f}s/epoch)')

# Save only the LoRA adapter -- much smaller than the full model
final_dir = OUTPUT_DIR / 'final'
ft_model.save_pretrained(str(final_dir))
ft_tokenizer.save_pretrained(str(final_dir))
print(f'Saved adapter to {final_dir}')

In [ ]:
# -- Cell 6b: Download adapter now (protects against session loss) --
# Run this immediately after training. If the session disconnects before
# Cell 11, you still have the adapter and can skip re-training.
import zipfile
from google.colab import files
from pathlib import Path

adapter_dir = Path('output/llama3_2_1b_qlora_finetune/final')
with zipfile.ZipFile('adapter.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in adapter_dir.rglob('*'):
        if f.is_file():
            z.write(f)

files.download('adapter.zip')
print('Adapter download started.')

---
## Section 4 -- Inference

### Baseline vs. fine-tuned comparison

We run inference with **two models** on the exact same test set:
1. **Baseline** -- `unsloth/Llama-3.2-1B-Instruct-bnb-4bit` with no fine-tuning
2. **Fine-tuned** -- the QLoRA adapter from Cell 6, loaded on top of the same base

Using the same test IDs ensures any difference in pass@1 is attributable to fine-tuning, not dataset variation.

### Why load baseline in 4-bit?

We load the baseline in 4-bit (matching training conditions) for a fair comparison. Loading the baseline in fp16 would give it a slight quality advantage and conflate model quality with precision effects.

### Generation settings

- **`temperature=0.2`** -- low temperature keeps output near-deterministic. Code generation benefits from this since correct Verilog syntax is narrow.
- **`max_new_tokens=1024`** -- generous budget for module implementations (typically 20-200 lines).
- **`do_sample=True`** -- enables temperature sampling. At 0.2 this is nearly greedy but avoids degenerate repetition loops.

### Why `limit=20` first (Cell 7b)?

Verifies the pipeline end-to-end (model loads, generates plausible Verilog, testbenches work) before committing to a full run that may take 30-60 minutes.

In [ ]:
# -- Cell 7: Inference helpers --
import json
import re
import torch
from pathlib import Path
from unsloth import FastLanguageModel

VERILOG_DB = Path('VerilogDB')
GENERATED_TESTS_DIR = Path('generated_tests')
DEFAULT_MODEL = 'unsloth/Llama-3.2-1B-Instruct-bnb-4bit'
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.2   # low temp = near-deterministic output
INFERENCE_MAX_SEQ = 2048


def build_prompt(spec):
    module_name = spec.get('module_name', 'unknown')
    description = spec.get('description', '').strip()
    ports = spec.get('ports', [])
    port_lines = []
    for p in ports:
        w = p.get('bit_width', 1)
        width_str = f'[{w-1}:0] ' if isinstance(w, int) and w > 1 else ''
        port_lines.append(f"  {p['direction']} {width_str}{p['name']}")
    port_str = '\n'.join(port_lines) if port_lines else '  (no ports listed)'
    return (
        f'Generate a Verilog module named `{module_name}` with the following specification.\n\n'
        f'Description: {description}\n\n'
        f'Ports:\n{port_str}\n\n'
        'Respond with only the complete Verilog module code. Do not include markdown fences or any explanation.'
    )


def extract_verilog(text):
    # Strip markdown fences if the model added them despite instructions
    if '```' in text:
        lines = text.split('\n')
        out, inside = [], False
        for line in lines:
            if line.strip().startswith('```'):
                inside = not inside
                continue
            if inside or not any(l.strip().startswith('```') for l in lines):
                out.append(line)
        text = '\n'.join(out)
    # Truncate at endmodule -- models sometimes hallucinate extra text after it
    m = re.search(r'\bendmodule\b', text, re.IGNORECASE)
    if m:
        end = text.find('\n', m.end())
        text = text[: end if end != -1 else m.end()]
    return text.strip()


def load_inference_model(model_name):
    # Load in 4-bit to match training conditions for a fair baseline comparison
    print(f'Loading {model_name} in 4-bit...')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=INFERENCE_MAX_SEQ,
        dtype=None,          # auto-detect bf16/fp16
        load_in_4bit=True,   # match training conditions
    )
    FastLanguageModel.for_inference(model)  # enables Unsloth's faster inference kernels
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    return model, tokenizer, device


def generate_one(model, tokenizer, device, prompt):
    messages = [{'role': 'user', 'content': prompt}]
    result = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,  # appends assistant turn start token
        return_tensors='pt',
    )
    input_ids = (result.input_ids if hasattr(result, 'input_ids') else result).to(device)
    attention_mask = torch.ones_like(input_ids)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            pad_token_id=tokenizer.pad_token_id,
        )
    # Slice off input tokens to get only newly generated tokens
    generated = output_ids[0, input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def run_inference(model_name, output_dir, ids_file=None, limit=None):
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    tb_files = sorted(GENERATED_TESTS_DIR.glob('*_tb.v'))
    if ids_file:
        # Restrict to test split -- prevents data leakage from train modules
        allowed = set(Path(ids_file).read_text(encoding='utf-8').splitlines())
        tb_files = [f for f in tb_files if f.stem.removesuffix('_tb') in allowed]
        print(f'Filtered to {len(tb_files)} testbenches from {ids_file}')
    if limit:
        tb_files = tb_files[:limit]

    print(f'Generating {len(tb_files)} modules -> {out_dir}')
    import time
    t_load = time.time()
    model, tokenizer, device = load_inference_model(model_name)
    print(f'Model loaded in {time.time() - t_load:.1f}s')
    t_infer_start = time.time()
    generated_count = 0

    skipped = 0
    for i, tb_path in enumerate(tb_files, 1):
        module_id = tb_path.stem.removesuffix('_tb')
        out_path = out_dir / f'{module_id}.v'
        if out_path.exists():
            skipped += 1
            continue
        spec_path = VERILOG_DB / f'{module_id}.json'
        if not spec_path.exists():
            print(f'[{i}/{len(tb_files)}] SKIP {module_id}: no spec')
            skipped += 1
            continue
        spec = json.loads(spec_path.read_text(encoding='utf-8'))
        prompt = build_prompt(spec)
        print(f'[{i}/{len(tb_files)}] Generating {module_id} ...', end=' ', flush=True)
        raw = generate_one(model, tokenizer, device, prompt)
        out_path.write_text(extract_verilog(raw), encoding='utf-8')
        generated_count += 1
        print('done')

    t_infer_elapsed = time.time() - t_infer_start
    avg = t_infer_elapsed / generated_count if generated_count else 0
    print(f'\nFinished. Generated {generated_count} modules, skipped {skipped}.')
    print(f'Inference time: {t_infer_elapsed:.1f}s total  ({avg:.1f}s/module avg)')

In [ ]:
# -- Cell 7b: Baseline inference -- 20-example smoke test --
# Verify the pipeline works before committing to the full run.
run_inference(
    model_name=DEFAULT_MODEL,
    output_dir='generated_verilog/baseline_test',
    ids_file='data/test_ids.txt',
    limit=20,
)

In [ ]:
# -- Cell 7c: Baseline inference -- full test set --
# Already-generated files from 7b are skipped automatically.
run_inference(
    model_name=DEFAULT_MODEL,
    output_dir='generated_verilog/baseline_test',
    ids_file='data/test_ids.txt',
)

In [ ]:
# -- Cell 8: Fine-tuned inference -- full test set --
# Unsloth merges the QLoRA adapter weights at inference time automatically.
run_inference(
    model_name='output/llama3_2_1b_qlora_finetune/final',
    output_dir='generated_verilog/finetuned_test',
    ids_file='data/test_ids.txt',
)

---
## Section 5 -- Evaluation

### What is pass@1?

**pass@k** is the standard metric for code generation: what fraction of problems does the model solve with k attempts? We use **pass@1** -- one attempt per problem -- the strictest variant.

A module passes if:
1. It compiles with `iverilog` (syntactically valid Verilog)
2. When simulated with `vvp`, the testbench output contains `PASS` and no `Error`/`Failed`/`FAIL`

### Failure categories

Each failed result is classified into exactly one category:

| Category | What happened |
|---|---|
| `compile_error` | `iverilog` rejected the code -- syntax error or undefined signal |
| `timeout` | `vvp` simulation did not finish within 30 s -- likely an infinite loop |
| `wrong_output` | Simulation ran but printed `Error`, `Failed`, or `FAIL` -- logic error |
| `no_output` | Simulation ran but never printed `PASS` -- structural testbench mismatch |
| `no_testbench` | No `*_tb.v` found for this module |

Tracking categories separately is more informative than a single pass/fail bit. For example, if fine-tuning reduces `compile_error` but not `wrong_output`, it improved syntax but not logic.

### Why not exact-match?

There are many valid implementations of the same hardware module. Exact-match would penalize correct but differently-structured code. Functional simulation is the right ground truth.

In [ ]:
# -- Cell 9: Evaluate both models --
import json
import re
import subprocess
from pathlib import Path

GENERATED_TESTS_DIR = Path('generated_tests')
SIM_DIR = Path('sim_build')
VVP_TIMEOUT_SEC = 30
FAIL_PATTERNS = re.compile(r'\b(Error:|Failed|FAIL)\b', re.IGNORECASE)

FAILURE_MODES = ['pass', 'compile_error', 'timeout', 'wrong_output', 'no_output', 'no_testbench']


def classify(msg):
    # pass          - compiled, ran, printed PASS, no errors
    # compile_error - iverilog rejected the code
    # timeout       - simulation ran forever
    # wrong_output  - simulation printed Error/Failed/FAIL
    # no_output     - simulation ran but never printed PASS
    # no_testbench  - no *_tb.v found
    if msg == 'pass':
        return 'pass'
    if 'no testbench' in msg:
        return 'no_testbench'
    if 'compile failed' in msg or 'iverilog not found' in msg or 'compile timeout' in msg:
        return 'compile_error'
    if 'timeout' in msg:
        return 'timeout'
    if 'Error/Failed' in msg:
        return 'wrong_output'
    return 'no_output'


def run_test(module_path, tb_path, rec_id):
    SIM_DIR.mkdir(parents=True, exist_ok=True)
    sim_binary = SIM_DIR / f'sim_{rec_id}'

    # Step 1: Compile with iverilog
    try:
        result = subprocess.run(
            ['iverilog', '-o', str(sim_binary), str(module_path), str(tb_path)],
            capture_output=True, text=True, timeout=15,
        )
    except FileNotFoundError:
        return False, 'iverilog not found'
    except subprocess.TimeoutExpired:
        return False, 'compile timeout'
    if result.returncode != 0:
        return False, f'compile failed: {result.stderr.strip()[:200]}'

    # Step 2: Simulate with vvp
    try:
        result = subprocess.run(
            ['vvp', str(sim_binary)],
            capture_output=True, text=True, timeout=VVP_TIMEOUT_SEC,
        )
    except subprocess.TimeoutExpired:
        return False, 'vvp timeout'
    except Exception as e:
        return False, str(e)

    stdout = result.stdout or ''
    stderr = result.stderr or ''
    if FAIL_PATTERNS.search(stdout) or FAIL_PATTERNS.search(stderr):
        return False, 'test reported Error/Failed in output'
    if 'PASS' not in stdout:
        snippet = (stdout + stderr).strip()[:200] or '(no output)'
        return False, f'no PASS in output: {snippet}'
    return True, 'pass'


def evaluate(verilog_dir, results_json):
    vdir = Path(verilog_dir)
    v_files = sorted(vdir.glob('*.v'))
    print(f'Evaluating {len(v_files)} modules from {vdir}')
    results = []
    for module_path in v_files:
        rec_id = module_path.stem
        tb_path = GENERATED_TESTS_DIR / f'{rec_id}_tb.v'
        if not tb_path.exists():
            print(f'  {rec_id}: SKIP (no testbench)')
            results.append({'id': rec_id, 'passed': False, 'category': 'no_testbench', 'msg': 'no testbench'})
            continue
        passed, msg = run_test(module_path, tb_path, rec_id)
        category = classify(msg)
        print(f"  {rec_id}: {'PASS' if passed else 'FAIL'}  [{category}]  {msg[:80]}")
        results.append({'id': rec_id, 'passed': passed, 'category': category, 'msg': msg})

    total = len(results)
    counts = {m: sum(1 for r in results if r['category'] == m) for m in FAILURE_MODES}

    # Rates exclude no_testbench from the denominator
    evaluable = total - counts['no_testbench']
    compiled  = evaluable - counts['compile_error']
    pass_at_1      = counts['pass']         / total     if total     else 0.0
    compile_rate   = compiled               / evaluable if evaluable else 0.0
    timeout_rate   = counts['timeout']      / evaluable if evaluable else 0.0
    wrong_out_rate = counts['wrong_output'] / evaluable if evaluable else 0.0
    no_output_rate = counts['no_output']    / evaluable if evaluable else 0.0

    print(f"\n{'Metric':<28} {'Value':>8}")
    print('-' * 38)
    print(f"{'Total evaluated':<28} {total:>8d}")
    print(f"{'Pass@1':<28} {pass_at_1:>8.3f}  ({counts['pass']}/{total})")
    print(f"{'Compile rate':<28} {compile_rate:>8.3f}  ({compiled}/{evaluable})")
    print(f"{'Timeout rate':<28} {timeout_rate:>8.3f}  ({counts['timeout']}/{evaluable})")
    print(f"{'Wrong output rate':<28} {wrong_out_rate:>8.3f}  ({counts['wrong_output']}/{evaluable})")
    print(f"{'No-PASS rate':<28} {no_output_rate:>8.3f}  ({counts['no_output']}/{evaluable})")

    out = Path(results_json)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps({
        'metrics': {
            'pass_at_1':      pass_at_1,
            'compile_rate':   compile_rate,
            'timeout_rate':   timeout_rate,
            'wrong_out_rate': wrong_out_rate,
            'no_output_rate': no_output_rate,
            'counts':         counts,
            'total':          total,
            'evaluable':      evaluable,
        },
        'per_problem': results,
    }, indent=2), encoding='utf-8')
    print(f'Results written to {out}')


import os
os.makedirs('results', exist_ok=True)

evaluate('generated_verilog/baseline_test',  'results/baseline_test.json')
evaluate('generated_verilog/finetuned_test', 'results/finetuned_test.json')

---
## Section 6 -- Results and Interpretation

### How to read the comparison

The key number is **pass@1 improvement** = finetuned pass@1 - baseline pass@1.

Expected outcomes for a 1B model fine-tuned with QLoRA on Verilog:
- **Baseline**: 5-20% pass@1 (small models struggle with Verilog syntax)
- **Fine-tuned**: 15-45% pass@1 (slightly lower ceiling than fp16 LoRA due to quantization noise)

### Reading the failure mode table

Look at how categories shift between baseline and fine-tuned:
- `compile_error` drops significantly -> fine-tuning learned valid Verilog syntax
- `wrong_output` increases -> model now generates syntactically valid but logically wrong code (progress!)
- `timeout` appears -> model learned sequential logic but produced feedback loops

### Both-fail breakdown

Modules that both models fail on, broken out by failure category, reveal how the model is failing even when it does not pass. A shift from `compile_error` to `wrong_output` is meaningful progress even if pass@1 stays the same.

### Comparing QLoRA vs. fp16 LoRA

If you run both notebooks on the same dataset, compare:
- pass@1 difference (QLoRA is typically 1-3 pp lower)
- compile rate (similar -- quantization affects output quality, not syntax validity)
- Training time and peak VRAM (QLoRA should be meaningfully faster/cheaper)

### Limitations

- Testbenches were generated by GPT-4o-mini and may not cover all edge cases
- pass@1 on a small test set has high variance -- interpret trends, not exact numbers
- QLoRA introduces ~1-3% quantization noise vs fp16 LoRA; results may vary between runs

In [ ]:
# -- Cell 10: Compare results --
import json
from collections import Counter

def load_results(path):
    with open(path) as f:
        return json.load(f)

b = load_results('results/baseline_test.json')
f = load_results('results/finetuned_test.json')
bm, fm = b['metrics'], f['metrics']

# -- Summary metrics table --
print(f"{'Metric':<28} {'Baseline':>10} {'Finetuned':>10} {'Delta':>10}")
print('-' * 62)

def row(label, key):
    bv, fv = bm[key], fm[key]
    delta = fv - bv
    print(f'{label:<28} {bv:>10.3f} {fv:>10.3f} {delta:>+10.3f}')

row('Pass@1',            'pass_at_1')
row('Compile rate',      'compile_rate')
row('Timeout rate',      'timeout_rate')
row('Wrong output rate', 'wrong_out_rate')
row('No-PASS rate',      'no_output_rate')

# -- Absolute failure counts --
print()
bc, fc = bm['counts'], fm['counts']
print(f"{'Failure mode':<25} {'Baseline':>10} {'Finetuned':>10} {'Delta':>10}")
print('-' * 57)
for mode in ['compile_error', 'timeout', 'wrong_output', 'no_output', 'no_testbench']:
    label = mode.replace('_', ' ').title()
    bv, fv = bc[mode], fc[mode]
    print(f'{label:<25} {bv:>10d} {fv:>10d} {fv-bv:>+10d}')

# -- Per-module outcome breakdown --
b_per = {r['id']: r for r in b['per_problem']}
f_per = {r['id']: r for r in f['per_problem']}
both_ids = set(b_per) & set(f_per)

gained   = [i for i in both_ids if not b_per[i]['passed'] and     f_per[i]['passed']]
lost     = [i for i in both_ids if     b_per[i]['passed'] and not f_per[i]['passed']]
both_ok  = [i for i in both_ids if     b_per[i]['passed'] and     f_per[i]['passed']]
both_bad = [i for i in both_ids if not b_per[i]['passed'] and not f_per[i]['passed']]

print(f'\nPer-module outcome (n={len(both_ids)}):')
print(f'  Both pass:       {len(both_ok):4d}')
print(f'  Finetuned gains: {len(gained):4d}  (baseline FAIL -> finetuned PASS)')
print(f'  Finetuned loses: {len(lost):4d}  (baseline PASS -> finetuned FAIL)')
print(f'  Both fail:       {len(both_bad):4d}')

if gained:
    print(f"\nGained: {gained[:10]}{'...' if len(gained) > 10 else ''}")
if lost:
    print(f"Lost:   {lost[:10]}{'...' if len(lost) > 10 else ''}")

# -- Failure mode shift among both-fail --
# Even when both models fail, the failure category may change.
# e.g. compile_error -> wrong_output means syntax improved but logic did not.
if both_bad:
    print(f'\nFailure modes among both-fail ({len(both_bad)} modules):')
    b_modes = Counter(b_per[i]['category'] for i in both_bad)
    f_modes = Counter(f_per[i]['category'] for i in both_bad)
    all_modes = sorted(set(b_modes) | set(f_modes))
    print(f"  {'Mode':<20} {'Baseline':>10} {'Finetuned':>10}")
    print('  ' + '-' * 42)
    for m in all_modes:
        bv, fv = b_modes.get(m, 0), f_modes.get(m, 0)
        print(f'  {m:<20} {bv:>10d} {fv:>10d}')

In [ ]:
# -- Cell 11: Download results and adapter --
import zipfile
from google.colab import files
from pathlib import Path

with zipfile.ZipFile('output.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in Path('results').glob('*.json'):
        z.write(f)
    for f in Path('output/llama3_2_1b_qlora_finetune/final').rglob('*'):
        if f.is_file():
            z.write(f)

files.download('output.zip')
print('Download started.')